# Chapter 3 · Learning instead of counting

Chapter 2 reached 2.48 by counting, then hit a wall: every extra character of memory makes
the table 65 times bigger. Neural networks get past that wall by **learning** their
numbers instead of counting them.

This chapter starts small. The model has exactly the same 65 × 65 table as the counting
bigram, but it fills the table in by trial and error, a little at a time. It ends up at
the same number. What is new is the *method*, and it is the method that scales all the way
up to ChatGPT.

| | |
|---|---|
| **Open box** | one step of training: `train_step` in `tinygpt/training.py` |
| **The number** | 4.17 → 2.48 again, this time by learning |
| **Time** | about an hour |
| **Python you need** | functions; a first look at PyTorch |
| **GPU** | not needed: this chapter's model is tiny |

**How this chapter works.** Five steps, in order:

1. **Predict** `everyone`: guess before you run anything. A wrong guess is useful: it shows you what you expected.
2. **Run** `everyone`: run the cells and compare with your guesses.
3. **Investigate** `everyone`: open the box. Read the code that just ran and answer a few questions.
4. **Modify** `most people`: change one thing, predict what will happen, then explain what did.
5. **Make** `optional challenge`: build this chapter's piece yourself. A check tells you when it's right.

Run a cell with **Shift+Enter**. If the notebook restarts, run the setup cell again first.

## Setup

In [ ]:
#@title Setup: run this cell first (about 30 seconds) { display-mode: "form" }
# Downloads tinyGPT (on Colab) and installs it. You don't need to read or change this cell.
import importlib, os, subprocess, sys
from pathlib import Path

def run(*command):
    done = subprocess.run(command, capture_output=True, text=True)
    if done.returncode != 0:
        raise RuntimeError(f"{' '.join(command)} failed:\n{done.stdout}{done.stderr}")

if "google.colab" in sys.modules:
    repo = Path("/content/tinyGPT")
    if not repo.exists():
        run("git", "clone", "-q", "https://github.com/cgpan/tinyGPT.git", str(repo))
    root = repo / "entry"
else:
    here = Path.cwd().resolve()
    root = next((p for p in [here, *here.parents] if (p / "src" / "tinygpt").is_dir()), None)
    if root is None:
        raise FileNotFoundError("Can't find the tinyGPT folder. Open this notebook from inside it, or use Colab.")
os.chdir(root)
run(sys.executable, "-m", "pip", "install", "-q", ".")
importlib.invalidate_caches()

import tinygpt
tinygpt.print_setup()

## 1 · Predict `everyone`

The learning model starts with every score in its table set to zero, which means every
next character is equally likely.

In [ ]:
guess_start = None              # its loss before any learning, at step 0?
guess_end = None                # after 2,000 steps of learning? (counting got 2.48)
guess_transformer_start = None  # an untrained *transformer* also knows nothing. Its loss at step 0?

## 2 · Run `everyone`

The settings for training live in a `Config`. Here: show the model 64 random windows of
32 characters at each step, take 2,000 steps, and measure the loss every 200.

In [ ]:
import torch

tok, train_ids, val_ids = tinygpt.load_data()
cfg = tinygpt.Config(block_size=32, batch_size=64, learning_rate=0.01, max_iters=2000, eval_interval=200)
model = tinygpt.BigramLM(cfg)
print(f"{tinygpt.count_params(model):,} numbers to learn: one score for every pair of characters")

Each training step does the same thing: let the model guess every next character in the
batch, measure the loss, and nudge all 4,225 numbers so the loss would have been a little
smaller. Run it and watch the loss.

In [ ]:
model, history = tinygpt.train(cfg, model=model)

In [ ]:
tinygpt.plot_history(history)
print("your guesses:", guess_start, guess_end)

Did learning find the same table that counting did? Compare a few probabilities from
chapter 2's counted table with the learned one:

In [ ]:
counted = tinygpt.CountBigram(tok.vocab_size).fit(train_ids)
learned = torch.softmax(model.table.weight, dim=1).detach().cpu()   # each row of scores -> probabilities
for a, b in [("t", "h"), (" ", "t"), (".", "\n"), ("q", "u")]:
    i, j = tok.stoi[a], tok.stoi[b]
    print(f"{b!r} after {a!r}:   counted {counted.probs()[i, j]:.3f}   learned {learned[i, j]:.3f}")

## 3 · Investigate `everyone`

### Open the box

All the learning happens in one short function. `train` calls it 2,000 times:

In [ ]:
tinygpt.show(tinygpt.train_step)

1. **`model(x, y)`**: the model guesses every next character in the batch and scores its
   guesses. What kind of thing is `loss`: a table, a list, or a single number?
2. **`loss.backward()`** works out, for each of the 4,225 numbers, which way it should
   move to make the loss smaller. That direction is called the *gradient*.
3. **`optimizer.step()`** moves every number a little in that direction. What decides how
   big "a little" is? (Look at `cfg`.)
4. **`optimizer.zero_grad()`**: gradients pile up unless they are cleared. Why clear them
   *before* `backward`, not after?
5. **`clip_grad_norm_`** caps the size of one step. What could go wrong without it?
6. In the comparison above, the learned model is *more* sure that "u" follows "q" (0.97)
   than the counts are (0.90). Why? (Hint: what did `alpha = 1` do to the counts?)

*Your explanation (double-click to edit):*

### Why does a transformer start at 4.33?

The learning bigram started at exactly 4.17, because all its scores started equal. Now
build the full transformer from chapter 5, untrained, and measure it the same way.
**Check your third guess.**

In [ ]:
torch.manual_seed(1337)
big = tinygpt.GPT(tinygpt.load_config("configs/base.yaml"))     # 10.7 million random numbers
big.eval()                                                      # measuring, not training: dropout off
x, y = tinygpt.get_batch(val_ids, block_size=128, batch_size=32)
with torch.no_grad():
    logits, loss = big(x, y)
probs = torch.softmax(logits, dim=-1)
print(f"untrained transformer: loss {loss.item():.2f}   (your guess: {guess_transformer_start})")
print(f"its favourite next character gets {probs.max(-1).values.mean():.1%} on average; "
      f"an even spread would give each {1 / 65:.1%}")

Knowing nothing *evenly* scores 4.17. The untrained transformer knows nothing
*unevenly*: its random starting numbers make it lean towards random characters, giving
its favourite about 4% instead of 1.5%. Leaning the wrong way is worse than not leaning
at all, so it starts above 4.17. Training soon fixes that: in chapter 5 the same model
is down to 2.43 after 250 steps.

## 4 · Modify `most people`

**A. The learning rate.** It sets the size of each step. **Predict** what happens with a
much smaller one (0.001) and much bigger ones (1 and 10). Then run all five. They take a
few seconds each.

In [ ]:
runs = {}
for lr in [0.001, 0.01, 0.1, 1.0, 10.0]:
    cfg_lr = tinygpt.Config(block_size=32, batch_size=64, learning_rate=lr, max_iters=2000, eval_interval=200)
    _, runs[f"learning rate {lr}"] = tinygpt.train(cfg_lr, model=tinygpt.BigramLM(cfg_lr))
tinygpt.plot_history(runs)

Which learning rates crawl? Which ones bounce around without settling? Why?

*Your explanation (double-click to edit):*

**B. Break it on purpose.** Here is a training step that forgets to clear the old
gradients. `train` accepts your own step function through `step_fn`. **Predict:** will
it still learn?

In [ ]:
def forgetful_step(model, x, y, optimizer):
    logits, loss = model(x, y)
    loss.backward()               # no optimizer.zero_grad() first!
    optimizer.step()
    return loss.item()

_, forgetful = tinygpt.train(cfg, model=tinygpt.BigramLM(cfg), step_fn=forgetful_step)
tinygpt.plot_history({"normal step": history, "forgot zero_grad": forgetful})

It gets *worse* the longer it trains. Each step's gradient is added to every earlier
one, so the model keeps pushing in old directions that no longer apply. Explain in your
own words:

*Your explanation (double-click to edit):*

## 5 · Make `optional challenge`

Write `train_step` yourself, from memory if you can. The check runs your function and
tinyGPT's side by side for three steps and compares the numbers in the table.

In [ ]:
def my_train_step(model, x, y, optimizer):
    """One step of learning. Return the loss on this batch as a plain number."""
    # 1. run the model on x and y to get (logits, loss)
    # 2. clear the old gradients
    # 3. work out the new gradients
    # 4. move every number a little
    return 0.0                    # replace this line: return the loss, using loss.item()

tinygpt.check.train_step(my_train_step)

Once the check passes, train the real model with your step instead of tinyGPT's:

In [ ]:
_, mine = tinygpt.train(cfg, model=tinygpt.BigramLM(cfg), step_fn=my_train_step)

**Stuck?** Try first, then compare with [the solution](https://github.com/cgpan/tinyGPT/blob/main/entry/solutions/03_learning.md) (also in the `solutions/` folder of your copy).

## Where the number stands

In [ ]:
tinygpt.scoreboard({"learning pairs": history[-1]["val_loss"]})

Learning landed on the same number as counting: about 2.49 against 2.48. Measured on the
whole validation text the two are 2.489 and 2.482. Same model, same answer. But learning
does not need a table for every possible context, so in chapter 4 the model can look much
further back.

## Common questions

**What is a gradient?** For each number in the model, the gradient says how the loss
would change if you nudged that number up a little. Training moves each number the other
way, downhill.

**If learning gives the same answer as counting, why bother?** Because counting needs a
cell for every context it has ever seen, and learning doesn't. In chapter 4 a model with
about 220,000 learned numbers uses 64 characters of memory; a counting table for that
would need a number of cells 118 digits long.

**What is AdamW?** The optimizer: the rule that turns gradients into steps. It is a
cleverer version of "step downhill" that adapts the step size for each number.

**Why does the loss wiggle instead of falling smoothly?** Each measurement uses 20 random
batches, and each training step sees only 64 random windows. Both add noise.

**What are `torch` and tensors?** PyTorch is the library that computes gradients for us.
A tensor is its version of a NumPy array; `.item()` turns a one-number tensor into an
ordinary Python number.

## If something breaks

| What you see | What to do |
|---|---|
| The loss becomes `nan` | The learning rate is far too big. Go back to 0.01. |
| `RuntimeError: Expected all tensors to be on the same device` | Something is on the GPU and something on the CPU. Move it with `.cpu()`, as in the comparison cell. |
| The check says the weights did not change | Your function never calls `optimizer.step()`. |
| `NameError` | Run the cells above it, in order. In Colab: Runtime > Run before. |

## Next

**Chapter 4 · Looking back: attention.** Give the model a way to use all the characters
before it, not just one. The number falls below 2.48 for the first time.